# Module 22: Distributed LLM Serving PagedAttention vLLM — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/llm_inference_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import llm_inference_engine

classes = [n for n, o in inspect.getmembers(llm_inference_engine, inspect.isclass)
           if o.__module__ == 'llm_inference_engine']
functions = [n for n, o in inspect.getmembers(llm_inference_engine, inspect.isfunction)
             if o.__module__ == 'llm_inference_engine']

print('module   : llm_inference_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(llm_inference_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Paged attention block allocation and freeing

This is the module's own `test_paged_attention_block_allocation_and_freeing` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from llm_inference_engine import (
    ContinuousBatchScheduler,
    InferenceRequest,
    PagedAttentionBlockManager,
    RequestStatus,
)

num_blocks = 10
block_size = 4
mgr = PagedAttentionBlockManager(num_gpu_blocks=num_blocks, block_size=block_size)

assert mgr.num_free_blocks() == 10
assert mgr.get_memory_utilization() == 0.0

# Request requiring 9 tokens -> ceil(9/4) = 3 blocks
req_id = "req_1"
ok = mgr.ensure_blocks_for_length(req_id, total_tokens=9)
assert ok is True
assert len(mgr.block_tables[req_id]) == 3
assert mgr.num_free_blocks() == 7
assert mgr.get_memory_utilization() == 0.3

# Free sequence blocks
mgr.free_blocks(req_id)
assert mgr.num_free_blocks() == 10
assert mgr.get_memory_utilization() == 0.0

print('PASSED: test_paged_attention_block_allocation_and_freeing')

## 3. 🔮 Prediction — commit before you run

Predict the KV-cache memory for 100 concurrent sequences of 2,000 tokens each, and what fraction PagedAttention reclaims from internal fragmentation.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_continuous_batching_lifecycle`, which tests exactly this property.


In [ ]:
mgr = PagedAttentionBlockManager(num_gpu_blocks=20, block_size=4)
scheduler = ContinuousBatchScheduler(block_manager=mgr, max_batch_size=2)

r1 = InferenceRequest("req_fast", "Hi", ["p1"], max_new_tokens=2)
r2 = InferenceRequest("req_slow", "Hello world", ["p1", "p2"], max_new_tokens=4)

scheduler.add_request(r1)
scheduler.add_request(r2)

assert not scheduler.is_idle()

# Step 1: both requests generate token 1
tokens_step1 = scheduler.step()
assert "req_fast" in tokens_step1
assert "req_slow" in tokens_step1

# Step 2: r1 generates token 2 and finishes!
scheduler.step()
assert r1.status == RequestStatus.FINISHED
assert r1.finish_step == 2
assert len(scheduler.finished_requests) == 1

# Add r3 while r2 is still running
r3 = InferenceRequest("req_new", "Test", ["p1"], max_new_tokens=1)
scheduler.add_request(r3)

# Step 3: r3 should be admitted immediately into running batch alongside r2
tokens_step3 = scheduler.step()
assert "req_new" in tokens_step3
assert "req_slow" in tokens_step3
assert r3.status == RequestStatus.FINISHED

# Step 4: r2 completes
scheduler.step()
assert r2.status == RequestStatus.FINISHED
assert scheduler.is_idle()

print('PASSED: test_continuous_batching_lifecycle')

## 4. Measure it: Gpu memory preemption handling

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_gpu_memory_preemption_handling` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

mgr = PagedAttentionBlockManager(num_gpu_blocks=2, block_size=2)
scheduler = ContinuousBatchScheduler(block_manager=mgr, max_batch_size=2)

r1 = InferenceRequest("req_1", "P", ["p1"], max_new_tokens=3)
r2 = InferenceRequest("req_2", "P", ["p1"], max_new_tokens=3)
scheduler.add_request(r1)
scheduler.add_request(r2)

# Step 1: Both admitted (each has 1 token -> 1 block each, 2 blocks total used)
scheduler.step()

# Step 2: One of the sequences expands to 3 tokens (crosses boundary -> requires 2 blocks)
# Total required is 3 blocks, but only 2 exist -> triggers preemption!
scheduler.step()

# At least one request should remain active or be preempted back to waiting queue
preempted = [r for r in scheduler.waiting_queue if r.status == RequestStatus.PREEMPTED]
assert len(preempted) >= 0  # Preemption mechanism handled gracefully without crashing

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_gpu_memory_preemption_handling')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(llm_inference_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. KV cache, not model weights, is what limits concurrent sequences.
2. Paged allocation reclaims the internal fragmentation that naive caching wastes.
3. Continuous batching keeps the GPU busy across requests of different lengths.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
